# Recommender Systems on Real Data: From Neighbours to Three Ways of Factorizing

## The situation

You have just joined the personalisation team at **Flix**, a streaming service with **943
subscribers** and **1,682 films**: a ledger of 1.6 million cells, **94% blank**. Every
homepage opens with a **"Top picks for you"** row of 10 tiles, today the same bestsellers
for everyone.

Mira, Flix's Head of Product, wants answers:

1. How do we **score** a recommender *before* we ship it?
2. Can "subscribers who liked this also liked…" beat the bestsellers?
3. Can a few hidden **taste scores** per subscriber and per film do better still?
4. Which homepage do we ship?

Fill in the 🎯 cells and run everything top to bottom. ▶ cells are plumbing: run them and
read the output.

In [ ]:
#@title 🗺️ Roadmap (double-click to view the code) { display-mode: "form" }
import uuid
from IPython.display import HTML, display


def roadmap():
    """Draws the notebook's route as two rows of numbered steps."""
    acts = [
        ("Part I · People who liked this also liked…", [
            ("🧸", "0", "Toy ledger"), ("📒", "1", "Real ledger"), ("🏆", "2", "Bestsellers & scores"),
            ("🔗", "3", "Neighbours + a rule"), ("🧱", "·", "Where it breaks")]),
        ("Part II · Hidden taste scores", [
            ("🗜️", "4a", "SVD"), ("🧩", "4b", "NMF"), ("⭐", "4c", "Embeddings"),
            ("🙋", "5", "Your homepage"), ("🚀", "6", "Ship it"),
            ("🧠", "⭐", "Bonus: GMF · GMF+MLP"), ("📎", "A", "Appendix: how P, Q are found")]),
    ]
    cid = "rm" + uuid.uuid4().hex[:8]
    rows = []
    for title, steps in acts:
        items = []
        for icon, n, label in steps:
            items.append(f'<div class="step"><div class="dot">{icon}</div>'
                         f'<div class="lab"><b>{n}</b> · {label}</div></div>')
        rows.append(f'<div class="act"><div class="title">{title}</div>'
                    f'<div class="steps">{"".join(items)}</div></div>')
    display(HTML(f"""
<style>
#{cid} {{font-family: system-ui, 'Segoe UI', Roboto, sans-serif; color:#222; border-radius:18px;
  border:1px solid #ecebff; background:linear-gradient(135deg,#f6f8ff,#fbf5ff); padding:14px 18px;}}
#{cid} .act {{background:#fff; border:1px solid #e6e8ee; border-radius:14px; padding:10px 12px; margin:8px 0;}}
#{cid} .title {{font-weight:700; color:#333; margin-bottom:8px; font-size:15px;}}
#{cid} .steps {{display:flex; flex-wrap:wrap; gap:14px;}}
#{cid} .step {{width:116px; text-align:center;}}
#{cid} .dot {{width:52px; height:52px; margin:0 auto; border-radius:50%; font-size:24px; line-height:52px;
  background:linear-gradient(135deg,#667eea,#764ba2); box-shadow:0 2px 6px rgba(102,126,234,.35);}}
#{cid} .lab {{font-size:13px; color:#333; margin-top:6px;}}
</style>
<div id="{cid}">{"".join(rows)}</div>"""))


roadmap()

In [ ]:
#@title Setup (run this first) { display-mode: "form" }
import base64, gzip, importlib, os, sys, uuid, warnings
warnings.filterwarnings("ignore")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")      # keep TensorFlow quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, display
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

PURPLE, PURPLE2 = '#667eea', '#764ba2'                 # primary / gradient accent
GREEN, RED, AMBER = '#39b36a', '#e0796d', '#e0a23c'    # good / bad / caution
RANDOM_STATE = 0

# The Flix engine (data loading, scoring, the homepage renderer, training loops) ships
# inside this cell, compressed. It is written to ./flix_engine/ and imported from there.
_ENGINE = "..."
os.makedirs("flix_engine", exist_ok=True)
with open("flix_engine/recsys_utils.py", "wb") as f:
    f.write(gzip.decompress(base64.b64decode(_ENGINE)))
sys.path.insert(0, os.path.abspath("flix_engine"))
import recsys_utils
importlib.reload(recsys_utils)
from recsys_utils import (Leaderboard, axes_report, because_you_liked, check_cosine, compare_neighbours,
                          coverage, film_map, film_menu, find_film, gmf_explainer, hits_at_10,
                          item_item_scores, k_sweep, keras_available, load_movielens, make_ledger,
                          more_like_this, my_homepages, nmf_axes_report, only_well_known, orient_axes,
                          plot_k_sweep, plot_training, require, rule_sweep, run_nmf, sgd_steps,
                          show_factorization, show_homepage, show_row, show_steps, show_table,
                          split_ratings, subscriber_profile, svd_steps, table_html, test_predictions,
                          train_embeddings, train_gmf, train_ncf, use_split)

layers = recsys_utils.keras_layers()      # Keras layers (None if TensorFlow is missing)
print("✓ Setup complete" + ("" if keras_available() else " · TensorFlow not found: the GMF bonus will be skipped"))

## Part 0 — The toy ledger: what makes two films similar?

**3 subscribers × 4 films**, rated 1★–5★, blank = not rated. Movies 1–2 are action, Movies
3–4 romance. User A loves action, User B romance (and has seen no action film), User C a
bit of both.

In [ ]:
#@title ▶ The toy ledger: three subscribers, four films { display-mode: "form" }
toy = pd.DataFrame(
    [
        [5,      5,      1,      np.nan],   # User A: loves action, dislikes romance
        [np.nan, np.nan, 5,      4],        # User B: loves romance, no action seen
        [3,      np.nan, 2,      5],        # User C: a bit of both
    ],
    index=["User A", "User B", "User C"],
    columns=["Movie 1 (Act)", "Movie 2 (Act)", "Movie 3 (Rom)", "Movie 4 (Rom)"],
)
show_table(toy, title="The toy ledger", subtitle="rows: subscribers · columns: films · blank = not rated",
           fmt="{:.0f}★")

### Film twins

Two films are *similar* if **the same people rated them the same way**. Two steps.

**Step 1: centre.** Subtract each subscriber's own average, so +1 means "one star above *my*
usual". This evens out generous and harsh raters. A blank becomes 0: "no opinion".

In [ ]:
#@title ▶ Step 1: centre each subscriber's ratings { display-mode: "form" }
toy_centred = toy.sub(toy.mean(axis=1), axis=0).fillna(0)
show_table(toy_centred, title="Each rating relative to the subscriber's own average", heat="div",
           subtitle="+ = above their usual · − = below · 0 = their usual, or no opinion")

**Step 2: cosine similarity** of two film columns, where $x_{u,a}$ is subscriber $u$'s
centred rating of film $a$:

$$\text{sim}(a, b) = \frac{\sum_u x_{u,a}\, x_{u,b}}{\sqrt{\sum_u x_{u,a}^2}\;\sqrt{\sum_u x_{u,b}^2}}$$

The top adds up "did each subscriber feel the same about both films?": same signs add,
opposite signs subtract. The bottom divides by the two columns' lengths, so the result runs
from **−1** (opposite audiences) through **0** (unrelated) to **+1** (twins).

## 🎯 Exercise 1. Cosine similarity

**🎯 Task.** Write `cosine(a, b)` for two columns of numbers.

### 🎯 Solution

In [ ]:
# 🎯 Exercise 1: cosine similarity between two columns of numbers.
def cosine(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    agreement = np.sum(a * b)
    lengths = np.sqrt(np.sum(a ** 2)) * np.sqrt(np.sum(b ** 2))
    return agreement / lengths

In [ ]:
#@title ▶ Every pair of toy films, with your function { display-mode: "form" }
check_cosine(cosine)
toy_similarity = pd.DataFrame(
    [[cosine(toy_centred[a], toy_centred[b]) for b in toy.columns] for a in toy.columns],
    index=toy.columns, columns=toy.columns)
show_table(toy_similarity, title="How similar are the toy films?", heat="div",
           subtitle="+1 = rated identically by the same people · 0 = unrelated · −1 = opposite tastes")

**Reading the result.** ✅ The action films are near-twins (**0.97**) and both are negatively
similar to Movie 3. Movie 2 vs Movie 4 is 0.00: nobody rated both, so there is no evidence.
The method found the genres from three people's ratings alone.

User C never rated Movie 2. The simplest recommender: use C's rating of Movie 2's twin.

In [ ]:
#@title ▶ Movie 2's twin, and a guess for User C { display-mode: "form" }
twin_of_movie_2 = toy_similarity["Movie 2 (Act)"].drop("Movie 2 (Act)").idxmax()
guess_c_movie_2 = toy.loc["User C", twin_of_movie_2]
print(f"Movie 2's twin: {twin_of_movie_2} "
      f"(similarity {toy_similarity.loc['Movie 2 (Act)', twin_of_movie_2]:.2f})")
print(f"Our guess for User C × Movie 2: {guess_c_movie_2:.0f}★")

**Reading the result.** Movie 2's twin is Movie 1, which C rated 3★, so we guess 3★. That
is the whole trick: **borrow the subscriber's opinion of a similar film.**

💭 **Let's think about User B.** Could we guess B × Movie 1 the same way?

In [ ]:
#@title ▶ User B's ratings { display-mode: "form" }
show_table(toy.loc[["User B"]], title="User B's ratings", fmt="{:.0f}★", subtitle="blank = not rated")

B has rated only romance films, and both are *negatively* similar to Movie 1: **nothing to
borrow**, so this recommender has no answer. Part II shows methods that do.

## Part 1 — The real ledger

**MovieLens 100k**: 100,000 ratings (1★–5★) by 943 people of 1,682 films, 1997–98
(GroupLens, University of Minnesota; Harper & Konstan, *ACM TiiS* 5(4), 2015).

In [ ]:
#@title ▶ Load the ratings { display-mode: "form" }
ratings, films = load_movielens()
print(f"{len(ratings):,} ratings · {ratings['subscriber_id'].nunique()} subscribers · {len(films):,} films")
show_row(table_html(ratings.head(6), title="ratings: one row per rating", index=False),
         table_html(films[["title", "year", "genres", "n_ratings"]].head(6)
                    .assign(year=lambda d: d["year"].map(lambda y: "" if pd.isna(y) else str(y)),
                            genres=lambda d: d["genres"].str.join(", ")),
                    title="films: one row per film", subtitle="indexed by film id"))

In [ ]:
#@title ▶ The ledger: subscribers × films { display-mode: "form" }
full_ledger = make_ledger(ratings)     # rows: subscribers · columns: film ids · blank = not rated
show_table(full_ledger.iloc[:8, :12], title="The ledger (top-left corner)", fmt="{:.0f}★",
           subtitle=f"{full_ledger.shape[0]} subscribers × {full_ledger.shape[1]:,} films · blank = not rated")

In [ ]:
#@title 📊 How empty, and how lopsided? { display-mode: "form" }
sparsity = full_ledger.isna().to_numpy().mean()
print(f"Blank cells: {sparsity:.1%} of {full_ledger.size:,}")

per_film = ratings["film_id"].value_counts().to_numpy()          # sorted, most-rated first
share = np.cumsum(per_film) / per_film.sum()
half = int(np.searchsorted(share, 0.5)) + 1

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.fill_between(np.arange(1, len(per_film) + 1), per_film, color=PURPLE, alpha=0.35)
ax.fill_between(np.arange(1, half + 1), per_film[:half], color=PURPLE)
ax.annotate(f"top {half} films ({half / len(films):.0%} of the catalogue)\nreceive half of all ratings",
            xy=(half, per_film[half]), xytext=(half + 120, per_film[0] * 0.6), fontsize=10,
            arrowprops=dict(arrowstyle="->", color="#555"))
ax.set_xlabel("films, ranked from most-rated to least-rated")
ax.set_ylabel("number of ratings")
ax.set_title("The long tail: a few films get most of the attention")
plt.tight_layout(); plt.show()
print(f"Films with a single rating: {(per_film == 1).sum()}")

**Reading the result.** **94% blank**, and lopsided: 215 films (13%) collect half of all
ratings, 141 films have exactly one. Every method below is a way of filling those blanks,
and the long tail keeps coming back.

## Part 2 — Two questions, two scores: are the bestsellers good enough?

A model can fit the ratings it has seen and still be useless on the ones it hasn't. So we
**hide 20% of the ratings**, let every method learn from the other 80%, and check its
guesses against the hidden 20%. Two questions, two scores:

* ⭐ **How many stars?** Score: **RMSE**, the typical size of a star-prediction error.
  Lower is better.
* 🏠 **Which 10 films open the homepage?** Score: **Hits@10**: of a method's top 10 unrated
  films, how many the subscriber went on to watch (hidden 20%) *and* rate 4★+. Higher is
  better. A subscriber typically has about 8 such films hidden, so 10 is unreachable:
  compare methods against each other.

In [ ]:
#@title 📊 Visualization: two questions, two scores (double-click to view the code) { display-mode: "form" }
cid = "tq" + uuid.uuid4().hex[:8]
display(HTML(f"""
<style>
#{cid} {{font-family: system-ui, 'Segoe UI', Roboto, sans-serif; color:#222; border-radius:18px; border:1px solid #ecebff;
  background:linear-gradient(135deg,#f6f8ff,#fbf5ff); padding:14px 18px;}}
#{cid} .row {{display:flex; flex-wrap:wrap; gap:12px;}}
#{cid} .card {{flex:1; min-width:260px; background:#fff; border:1px solid #e6e8ee; border-radius:14px; padding:10px 14px;}}
#{cid} .bar {{display:flex; height:30px; border-radius:8px; overflow:hidden; margin:6px 0 12px 0; font-size:13px; color:#fff; font-weight:600;}}
#{cid} .bar div {{display:flex; align-items:center; justify-content:center;}}
#{cid} h4 {{margin:2px 0 6px 0; font-size:16px; color:{PURPLE2};}}
#{cid} p {{margin:4px 0; font-size:14px; color:#333;}}
</style>
<div id="{cid}">
  <b style="font-size:15px; color:#222;">100,000 known ratings</b>
  <div class="bar"><div style="width:80%; background:{PURPLE};">80% · methods learn from these</div>
  <div style="width:20%; background:{AMBER};">20% · hidden</div></div>
  <div class="row">
    <div class="card"><h4>⭐ How many stars?</h4>
      <p>Predict each hidden rating. Compare with the truth.</p>
      <p><b>RMSE</b> ≈ "on average we're off by this many stars". <span style="color:{GREEN};"><b>Lower is better.</b></span></p>
      <p style="color:#666;">Matters when you show a "you'll rate this 4.5★" badge.</p></div>
    <div class="card"><h4>🏠 What goes on the homepage?</h4>
      <p>Pick 10 films the subscriber hasn't rated. Count those they watched in the hidden 20% <i>and</i> rated 4★+.</p>
      <p><b>Hits@10</b> = good tiles out of 10. <span style="color:{GREEN};"><b>Higher is better.</b></span></p>
      <p style="color:#666;">Matters for the "Top picks for you" row.</p></div>
  </div>
</div>"""))

In [ ]:
#@title ▶ Hide 20% of the ratings, and meet two subscribers { display-mode: "form" }
train, test = split_ratings(ratings, test_share=0.2, seed=RANDOM_STATE)
use_split(train, test, films)              # registers the split with the scoring functions
ledger = make_ledger(train)                # the ledger every method learns from (blank = unknown)
board = Leaderboard()                      # one scoreboard for the whole notebook

NORA, ELIAS = 541, 585                     # two subscribers we'll follow throughout
SHOWCASE = {NORA: "Nora", ELIAS: "Elias"}
print(f"Learning from {len(train):,} ratings · {len(test):,} hidden for scoring")
for s, name in SHOWCASE.items():
    subscriber_profile(s, name)

Two subscribers we follow throughout: **Nora** (541) likes warm, family-friendly films;
**Elias** (585) period drama and European arthouse. A good personal row should differ for
the two.

## 🎯 Exercise 2. The star-error score

For hidden ratings $r_j$ and predictions $\hat r_j$, $j = 1, \dots, n$, the **root mean
squared error** squares each miss (big misses count extra), averages, and takes the root to
get back to stars:

$$\text{RMSE} = \sqrt{\frac{1}{n} \sum_{j=1}^{n} \left(r_j - \hat r_j\right)^2}$$

**🎯 Task.** Write `rmse(actual, predicted)`.

### 🎯 Solution

In [ ]:
# 🎯 Exercise 2: the star-error score.
def rmse(actual, predicted):
    errors = np.asarray(actual, dtype=float) - np.asarray(predicted, dtype=float)
    return np.sqrt(np.mean(errors ** 2))

In [ ]:
#@title ▶ Baseline: the overall average, scored { display-mode: "form" }
# Baseline 0: predict the overall average rating for everyone and everything.
overall_average = train["rating"].mean()
print(f"Overall average rating: {overall_average:.2f}★")
board.add("Predict the overall average for everyone",
          rmse=rmse(test["rating"], np.full(len(test), overall_average)))

✅ About **1.13 stars**: the overall average is off by roughly one star. Every star predictor
must beat that.

## 🎯 Exercise 3. Which bestseller list?

A homepage that is the same for everyone is a **ranking rule**: sort every film by one
number, show the top 10. The number can be how often the film was watched (**most watched**)
or its average stars (**best rated**). Taken literally, best-rated looks like this:

In [ ]:
#@title ▶ The best-rated films, taken literally { display-mode: "form" }
film_stats = train.groupby("film_id")["rating"].agg(["count", "mean"])
top_by_mean = film_stats.sort_values("mean", ascending=False).head(8).join(films["title"])
show_table(top_by_mean, title="The 8 best-rated films, taken literally",
           subtitle="count = number of ratings · mean = average stars")

Obscure titles with one or two 5★ ratings. A film rated once is not a safe bet: one
enthusiast tells us nothing about what the next thousand subscribers will think. A 4.3★
average over 300 ratings is far stronger evidence than 5.0★ over one, so an average should
only count once enough ratings stand behind it.

**🎯 Task.** Choose the minimum number of ratings a film needs to appear on a best-rated
list, and build that list.

### 🎯 Solution

In [ ]:
# 🎯 Exercise 3: a best-rated list that only trusts films with enough ratings.
MIN_RATINGS = 50
best_rated = film_stats.loc[film_stats["count"] >= MIN_RATINGS, "mean"]

In [ ]:
#@title ▶ Score the bestseller lists, show homepage v1 { display-mode: "form" }
require(MIN_RATINGS, best_rated, exercise="Exercise 3")
film_average = film_stats["mean"].reindex(ledger.columns).fillna(overall_average)
most_watched = film_stats["count"]

board.add("Rank by the film's average rating",
          rmse=rmse(test["rating"], test_predictions(film_average)),
          hits=hits_at_10(film_average))
board.add(f"Rank by average rating, ≥ {MIN_RATINGS} ratings", hits=hits_at_10(best_rated))
board.add("Rank by most watched (today's homepage)", hits=hits_at_10(most_watched))

board.show()
show_homepage(NORA, most_watched, "Homepage v1 · Most watched (the same for everyone)");

**Reading the result.** ✅ With `MIN_RATINGS = 50`:

| | RMSE ⭐ | Hits@10 🏠 |
|---|---|---|
| Rank by the film's average rating | 1.02 | **0.00** |
| Rank by average rating, ≥ 50 ratings | | 0.66 |
| Rank by most watched | | **1.36** |

Two lessons for Mira, Flix's Head of Product:

* **A good star predictor can make a terrible homepage.** The film average beats the overall
  average on stars, yet its top 10 is one-rating obscurities: 0 hits.
* **"Most watched" beats "best rated".** People *choose* popular films. Today's row scores
  about 1.4 good tiles out of 10; that is the number to beat.

## Part 3 — "More like this", "Top picks for you", and a business rule

Part 0 at full scale: centre each subscriber's ratings, then cosine similarity between every
pair of film columns. With 1,682 films that is 1.4 million pairs, so we use scikit-learn's
`cosine_similarity`, the same formula as your `cosine()` for every pair at once.

In [ ]:
centred = ledger.sub(ledger.mean(axis=1), axis=0).fillna(0)       # the toy's Step 1, on 943 rows
film_similarity = cosine_similarity(centred.T)                     # Exercise 1, for every pair of films
print(f"Similarity table: {film_similarity.shape[0]:,} × {film_similarity.shape[1]:,} films")
more_like_this(["Star Wars (1977)", "Toy Story (1995)"], film_similarity);

**Reading the result.** *Star Wars* pulls up its sequels and *Raiders*; *Toy Story* other
family favourites. Nobody told the method about genres or sequels: **it found them from who
rated what**. That is collaborative filtering, and for "More like this" under a popular film
it is already shippable. Try `find_film("matrix")` and `more_like_this(...)` on a film you
know.

### "Top picks for you" from neighbouring films

A personal row starts from a subscriber: for every film $f$ they haven't rated, take the 20
films **they did rate** that are most similar to $f$ (the *neighbours* $N$) and see how much
they liked those:

$$\hat r_{u,f} = \bar r_u + \frac{\sum_{g \in N} \text{sim}(f,g)\,\big(r_{u,g} - \bar r_u\big)}{\sum_{g \in N} \left|\text{sim}(f,g)\right|}$$

$\bar r_u$ is subscriber $u$'s average rating, $r_{u,g}$ their rating of neighbour $g$. In
words: **your usual rating, plus a similarity-weighted average of how far above or below
usual you rated the neighbours.** The $|\cdot|$ keeps the weights' total positive.

## 🎯 Exercise 4. The prediction rule

The engine finds the neighbours; you write the rule. For one subscriber your function
receives `subscriber_average` (one number) and two tables of the same shape, **one row per
film to score, one column per neighbour** (1,682 × 20): `similarities` and
`neighbour_ratings`.

Two NumPy facts are all you need. `similarities * deviations` multiplies the two tables
**cell by cell** (row 7, column 3 of one times row 7, column 3 of the other), so the result
is again 1,682 × 20. The formula then needs two totals *per film*: `.sum(axis=-1)` means
**"add up along each row"**, turning a 1,682 × 20 table into 1,682 numbers, one per film.

### 🎯 Solution

In [ ]:
# 🎯 Exercise 4: predict a rating from the neighbouring films.
def predict_from_neighbours(subscriber_average, similarities, neighbour_ratings):
    deviations = neighbour_ratings - subscriber_average          # how far above/below their usual
    weighted_sum = (similarities * deviations).sum(axis=-1)      # one total per film
    total_weight = np.abs(similarities).sum(axis=-1)             # one total per film
    return subscriber_average + weighted_sum / total_weight

In [ ]:
#@title ▶ Score item-item, show homepage v2 for both subscribers { display-mode: "form" }
K_NEIGHBOURS = 20
item_item = item_item_scores(film_similarity, predict_from_neighbours, k=K_NEIGHBOURS)   # ~5 s
board.add("Item-item neighbours",
          rmse=rmse(test["rating"], test_predictions(item_item)),
          hits=hits_at_10(item_item))
for s, name in SHOWCASE.items():
    show_homepage(s, item_item, f"Homepage v2 · Item-item top picks for {name}",
                  captions=because_you_liked(s, film_similarity), badge_format="{:.1f}★")

**Reading the result.** ✅ RMSE ≈ **0.93**, the best so far on stars. Hits@10 ≈ **0.38**,
*worse* than the bestseller row (1.36): the tiles are films almost nobody has heard of, each
predicted at 5★. The rule answers "how much would you like it *if* you watched it", and for
an obscure film two agreeing viewers look like a perfect neighbour. The homepage asks a
different question: what will you actually pick?

## 🎯 Exercise 5. A business rule: only well-known films on the homepage

Nobody ships a model naked. The standard fix: **a film needs a minimum number of ratings to
be shown.** The table applies it to the item-item predictions for several thresholds, with
the hits and how many different films ever reach *anyone's* homepage.

In [ ]:
#@title 📊 The rule at several thresholds { display-mode: "form" }
show_table(rule_sweep(item_item), title="Item-item + \"well-known films only\"",
           subtitle="min. ratings a film needs to be eligible · 0 = no rule", heat=None)

**🎯 Task.** Choose Flix's threshold, `HOMEPAGE_MIN_RATINGS`. There is no single right
answer: hits against catalogue breadth.

### 🎯 Solution

In [ ]:
# 🎯 Exercise 5: the minimum number of ratings a film needs to be shown on the homepage.
HOMEPAGE_MIN_RATINGS = 50

In [ ]:
#@title ▶ Score the rule, show the row it produces { display-mode: "form" }
require(HOMEPAGE_MIN_RATINGS, exercise="Exercise 5")
item_item_rule = only_well_known(item_item, HOMEPAGE_MIN_RATINGS)
board.add(f"Item-item + rule (≥ {HOMEPAGE_MIN_RATINGS} ratings)", hits=hits_at_10(item_item_rule))
show_homepage(NORA, item_item_rule, "Homepage v2b · Item-item with the rule, for Nora",
              captions=because_you_liked(NORA, film_similarity), badge_format="{:.1f}★");

**Reading the result.** ✅ At 50, hits more than double (0.38 → **0.84**) and the row looks
like a product again, yet still below the bestsellers' 1.36. At 200 you get 1.32 hits but
only 68 different films ever shown: **the rule has turned the personal row back into the
bestseller list.** Mira owns that trade-off: breadth of catalogue against score.

## Where neighbours break

A film with 3 ratings, through "More like this":

In [ ]:
#@title ▶ "More like this" for a film almost nobody rated { display-mode: "form" }
niche_id = film_stats.index[film_stats["count"] == 3][0]
print(f"{films.loc[niche_id, 'title']}: rated by {film_stats.loc[niche_id, 'count']} subscribers")
more_like_this(films.loc[niche_id, "title"], film_similarity);

Similarities near 1.0 with a random-looking set: three ratings are not evidence. Three
limits, each a business problem:

1. **Thin evidence.** At 94% blank, niche films rarely share viewers; their similarities are
   noise, and without the rule they flood the row.
2. **Nothing to borrow.** User B × Movie 1: a method that copies opinions of *similar* films
   is silent on a film unlike anything you watched.
3. **Scale.** 1,682 films is 2.8 million pairs; a 50,000-title catalogue is 2.5 *billion*.

One idea fixes all three: describe each subscriber and each film with **a handful of
numbers** and predict from those. Part II does that, three ways.

## Part 4 — Three ways to get P and Q

### The idea: a few hidden taste scores

Give every subscriber $k$ numbers, their **taste scores**, and every film $k$ numbers, its
**profile**. Predict any pair by multiplying number by number and adding up, a *dot product*:

$$\text{score}(u, f) = p_{u,1}\, q_{f,1} + \dots + p_{u,k}\, q_{f,k}$$

$p_{u,i}$ is subscriber $u$'s score on **axis** $i$, $q_{f,i}$ film $f$'s. Same signs on an
axis push the score up, opposite signs down.

Stack the subscribers' lists into a table $P$ (one row each, $k$ columns) and the films'
into $Q$ ($k$ rows, one column each). Every prediction at once is the matrix product

$$R \approx P \times Q$$

where $R$ is the ledger. Finding a good $P$ and $Q$ is **factorizing** $R$: with $k = 15$,
about 40,000 numbers explain 1.6 million cells. That compression is the point: $P$ and $Q$
can only hold *patterns*, so they fill the blanks with patterns.

**Which table?** For the 🏠 question we factorize the **watch table**: 1 if the subscriber
rated the film (chose to watch it), 0 if not. No stars in, so no stars out: read the scores
as *"how likely to pick this film"*. The toy, and the $P$ and $Q$ that SVD (next) finds for
it with $k = 2$:

In [ ]:
#@title ▶ The toy watch table, and the P and Q SVD finds for it { display-mode: "form" }
toy_watch = toy.notna().astype(int)        # 1 = rated it (chose to watch), 0 = didn't

toy_svd = TruncatedSVD(n_components=2, random_state=RANDOM_STATE)
toy_P, toy_Q = orient_axes(toy_svd.fit_transform(toy_watch), toy_svd.components_)
toy_P = pd.DataFrame(toy_P, index=toy.index, columns=["axis 1", "axis 2"])
toy_Q = pd.DataFrame(toy_Q, index=["axis 1", "axis 2"], columns=toy.columns)
show_factorization(toy_watch, toy_P, toy_Q, title="The toy watch table, and the P and Q that SVD finds for it (k = 2)",
                   subtitle="R: 1 = watched, 0 = not · P: two taste scores per subscriber · Q: two per film · "
                            "P × Q: the table rebuilt from those numbers",
                   row_labels=list(toy.index), col_labels=["M1 (Act)", "M2 (Act)", "M3 (Rom)", "M4 (Rom)"],
                   target_label="R · the toy watch table")

In [ ]:
#@title 🧩 Visualization: a prediction is a dot product (double-click to view the code) { display-mode: "form" }
cid = "dp" + uuid.uuid4().hex[:8]
p = toy_P.loc["User A"]
q = toy_Q["Movie 1 (Act)"]
b = (toy_P.loc["User B"] * toy_Q["Movie 1 (Act)"]).sum()
terms = " + ".join(f"({p[a]:.2f} × {q[a]:.2f})" for a in p.index)
display(HTML(f"""
<style>
#{cid} {{font-family: system-ui, 'Segoe UI', Roboto, sans-serif; color:#222; border-radius:18px; border:1px solid #ecebff;
  background:linear-gradient(135deg,#f6f8ff,#fbf5ff); padding:14px 18px;}}
#{cid} .row {{display:flex; flex-wrap:wrap; gap:12px; align-items:stretch;}}
#{cid} .card {{flex:1; min-width:180px; background:#fff; border:1px solid #e6e8ee; border-radius:14px; padding:10px 14px; font-size:14px;}}
#{cid} .card > b {{color:{PURPLE2};}}
#{cid} .v {{font-family: ui-monospace, Menlo, Consolas, monospace; font-size:15px;}}
#{cid} .eq {{margin-top:12px; background:#fff; border:1px solid #e6e8ee; border-radius:14px; padding:10px 14px; font-size:15px;}}
</style>
<div id="{cid}">
 <div class="row">
  <div class="card"><b>👤 User A's taste scores</b><div class="v">axis 1: {p['axis 1']:.2f}<br>axis 2: {p['axis 2']:.2f}</div></div>
  <div class="card"><b>🎬 Movie 1's profile</b><div class="v">axis 1: {q['axis 1']:.2f}<br>axis 2: {q['axis 2']:.2f}</div></div>
  <div class="card"><b>🧭 How to read the axes</b><br>axis 1: every film is positive, so it's an overall
     "watches a lot / gets watched a lot" score.<br>axis 2: <b>+ romance</b> · <b>− action</b></div>
 </div>
 <div class="eq">score(User A, Movie 1) = <span class="v">{terms}</span> = <b>{(p * q).sum():.2f}</b>
   &nbsp;→ close to 1: A did watch Movie 1, and two numbers per side reproduce that.<br>
   score(User B, Movie 1) = <b>{b:.2f}</b> → low: B's romance scores sit at the other end of axis 2 from Movie 1.
   <i>This is the cell the neighbours had no answer for.</i></div>
</div>"""))

Every method below produces a $P$ and a $Q$; they differ in **how** and in **which table**
they learn from. All are scored on the same hidden 20%.

In [ ]:
#@title 🧩 Visualization: the three methods at a glance (double-click to view the code) { display-mode: "form" }
cid = "fw" + uuid.uuid4().hex[:8]
rungs = [
    ("4a", "SVD", "Exact linear algebra: adds the best possible axis, then the next, …", "watch table (1 = watched)", "the homepage"),
    ("4b", "NMF", "Repeated small corrections; every number in P and Q stays ≥ 0.", "watch table", "readable axes"),
    ("4c", "Learned embeddings", "Predict one known rating, compare, nudge P and Q; plus a subscriber bias and a film bias.", "known stars only", "star badges"),
]
cards = "".join(
    f'<div class="card"><div class="tag">{n}</div><b>{t}</b><p>{d}</p>'
    f'<p class="meta">📒 learns from: {src}</p><p class="meta">🎯 good for: {good}</p></div>'
    for n, t, d, src, good in rungs)
display(HTML(f"""
<style>
#{cid} {{font-family: system-ui, 'Segoe UI', Roboto, sans-serif; color:#222; border-radius:18px; border:1px solid #ecebff;
  background:linear-gradient(135deg,#f6f8ff,#fbf5ff); padding:14px 18px; display:flex; flex-wrap:wrap; gap:10px;}}
#{cid} .card {{flex:1; min-width:170px; background:#fff; border:1px solid #e6e8ee; border-radius:14px; padding:10px 12px;}}
#{cid} .card > b {{color:{PURPLE2}; font-size:15px;}}
#{cid} .tag {{display:inline-block; font-size:11px; font-weight:700; color:#fff; border-radius:8px; padding:1px 7px;
  margin-bottom:4px; background:linear-gradient(135deg,{PURPLE},{PURPLE2});}}
#{cid} p {{margin:5px 0 0 0; font-size:13.5px; color:#333;}}
#{cid} .meta {{color:#666; font-size:12.5px;}}
</style>
<div id="{cid}">{cards}</div>"""))

### 4a. SVD: one axis at a time

**SVD** (*singular value decomposition*) finds the single axis that explains the most of the
table, then the best second axis for what is left, and so on. Stop after $k$ and you have
the best $P$ and $Q$ with $k$ axes, by a theorem. To watch it rebuild the toy table axis by
axis, see **Appendix A1**. Now the real watch table.

In [ ]:
#@title ▶ The real watch table { display-mode: "form" }
watch_table = ledger.notna().astype(int)        # 1 = watched, 0 = not
show_table(watch_table.iloc[:6, :14], title="The watch table (top-left corner)",
           subtitle=f"{watch_table.shape[0]} subscribers × {watch_table.shape[1]:,} films · 1 = watched")

## 🎯 Exercise 6. Factorize

`TruncatedSVD.fit_transform(table)` returns $P$ (943 × $k$); its `components_` attribute is
$Q$ ($k$ × 1,682). Their product is the full table of homepage scores, as in the toy.

**🎯 Task.** Complete `factorize(table, k)`.

### 🎯 Solution

In [ ]:
# 🎯 Exercise 6: k taste scores per subscriber (P), k per film (Q).
def factorize(table, k):
    svd = TruncatedSVD(n_components=k, random_state=RANDOM_STATE)
    subscriber_scores = svd.fit_transform(table)   # P: 943 × k
    film_scores = svd.components_                  # Q: k × 1,682
    return subscriber_scores, film_scores

## 🎯 Exercise 7. Choose the number of axes

The cell tries $k$ from 1 to 200 and scores each on the 🏠 question (about 10 seconds).

In [ ]:
#@title 📊 Try 1 to 200 taste axes { display-mode: "form" }
sweep = k_sweep(factorize, watch_table)
plot_k_sweep(sweep)

**🎯 Task.** Read the curve and choose `CHOSEN_K`.

### 🎯 Solution

In [ ]:
# 🎯 Exercise 7: the number of taste axes Flix will use.
CHOSEN_K = 15

In [ ]:
#@title ▶ Factorize with your k, score it, show homepage v3 { display-mode: "form" }
require(CHOSEN_K, exercise="Exercise 7")
subscriber_scores, film_scores = orient_axes(*factorize(watch_table, CHOSEN_K))
svd_scores = subscriber_scores @ film_scores
board.add(f"SVD on the watch table (k = {CHOSEN_K})", hits=hits_at_10(svd_scores))
for s, name in SHOWCASE.items():
    show_homepage(s, svd_scores, f"Homepage v3 · SVD top picks for {name}")

**Reading the result.** ✅ Hits rise from ~1.4 at 1 axis to **≈ 2.6 around 10–20**, then fall
to ~1 at 200: too few axes can't tell tastes apart, too many memorise individual quirks
(*overfitting*). The solution uses 15.

* 🏠 **Almost twice the bestseller row** (2.6 vs 1.36), and personal: family classics for
  Nora, period drama for Elias.
* ⭐ No RMSE, on purpose: these are "will pick" scores, not stars. Never put them on a badge.

**What people choose to watch is information.** A table with no stars in it makes the best
homepage so far.

#### What did the axes learn?

Axis 1 is popularity, as in the toy. The ends of axes 2 and 3:

In [ ]:
#@title ▶ The two ends of axes 2 and 3 { display-mode: "form" }
axes_report(film_scores, axes=(2, 3), label="SVD axis", title="What the axes learned: the two ends of axes 2 and 3",
            subtitle="well-known films only (≥ 50 ratings) · each axis is one row: lowest scores on the left, highest on the right")

💭 **Name the two axes.** One reading: axis 2 runs from **classics of past decades** to
**that season's releases**; axis 3 from **action blockbusters** to **critics' favourites and
arthouse**. Nobody gave the model genres or years. For Mira that is a second payoff: **the
axes describe the audience**, usable by marketing and content acquisition. The map below puts
the whole catalogue on those two axes at once.

In [ ]:
#@title 📊 The catalogue on axes 2 and 3 { display-mode: "form" }
film_map(film_scores, axis_x=2, axis_y=3,
         names_x=("classics of past decades", "that season's releases"),
         names_y=("action blockbusters", "critics' favourites, arthouse"))

### 4b. NMF: the same idea by repeated correction, no negative numbers

**NMF** (*non-negative matrix factorization*) insists every number in $P$ and $Q$ is ≥ 0, so
an axis reads as *how much of a theme* a film has, and a subscriber's row as *how much of
each theme they watch*. The price: no exact formula. NMF **starts from random positive
numbers and corrects them, over and over**, with one rule for $Q$ and its mirror for $P$
($\odot$ and the fraction are cell by cell):

$$Q_{\text{new}} = Q \;\odot\; \frac{P^{\mathsf T} R}{P^{\mathsf T} P\, Q}
\qquad\qquad
P_{\text{new}} = P \;\odot\; \frac{R\, Q^{\mathsf T}}{P\, Q\, Q^{\mathsf T}}$$

Each ratio compares **what the data asks** (top) with **what the current model gives**
(bottom): above 1 where the model undershoots, below 1 where it overshoots, so multiplying
nudges every entry the right way and positive stays positive. Why the ratio has exactly
this form, and your step watched on the toy: **Appendix A2**.

## 🎯 Exercise 8. One NMF correction step

**🎯 Task.** Write `nmf_step(R, P, Q)`: apply both rules once, return the new `P` and `Q`.
Every operation you need is in the cell's comments.

### 🎯 Solution

In [ ]:
# 🎯 Exercise 8: one NMF correction step, first Q then P.
def nmf_step(R, P, Q):
    eps = 1e-9                                        # avoids dividing by exactly zero
    Q = Q * (P.T @ R) / (P.T @ P @ Q + eps)           # Q ⊙ (what the data asks) / (what the model gives)
    P = P * (R @ Q.T) / (P @ Q @ Q.T + eps)           # same for P, using the freshly updated Q
    return P, Q

In [ ]:
#@title ▶ Your NMF on the real watch table: score it, read the axes { display-mode: "form" }
require(CHOSEN_K, exercise="Exercise 7")
nmf_P, nmf_Q, _ = run_nmf(nmf_step, watch_table, k=CHOSEN_K, iterations=200)       # ~5 s
nmf_scores = nmf_P @ nmf_Q
board.add(f"NMF on the watch table (k = {CHOSEN_K}, yours)", hits=hits_at_10(nmf_scores))
nmf_axes_report(nmf_Q, axes=(1, 2, 3, 4))
show_homepage(ELIAS, nmf_scores, "Homepage v3b · NMF top picks for Elias");

**Reading the result.** ✅ Your two-line NMF lands close to SVD on the homepage (see the
board), and its axes read as **themes without a "minus" end**: each card is a coherent shelf.
When the audience description matters as much as the hits, non-negative axes are easier to
put on a slide.

### 4c. Learned embeddings: back to stars, one rating at a time

For a "you'll rate this 4.3★" badge we need stars, and blanks must stay unknown. So $P$ and
$Q$ are learned **from the known ratings only**: start from random taste scores, take one
known rating, predict it with the dot product, nudge that subscriber's row of $P$ and that
film's column of $Q$ to shrink the error, and repeat for every known rating, many passes.
The learned scores are **embeddings**. The blanks are never looked at; their predictions
come from the patterns in the known cells. The loop step by step on the toy: **Appendix A3**.

On the real ledger, two additions:

* **Biases.** A **subscriber bias** $b_u$ (some people rate everything higher) and a **film
  bias** $b_f$ (some films are rated well by everyone), one learned number each.
* **The hidden 20%.** On the toy the known-cell error goes to zero if you keep going. On the
  real ledger we also watch the *hidden* error and stop when it stops improving.

$$\hat r_{u,f} = \mu + b_u + b_f + \underbrace{p_{u,1}\,q_{f,1} + \dots + p_{u,k}\,q_{f,k}}_{\text{taste match: the dot product}}$$

$\mu$ is the overall average rating.

## 🎯 Exercise 9. What is a rating made of?

**🎯 Task.** Write the formula as a function. The engine calls it on batches of ratings, so
sum the taste match with `axis=-1` ("add up each row"), as in Exercise 4.

### 🎯 Solution

In [ ]:
# 🎯 Exercise 9: a rating = average + subscriber bias + film bias + taste match.
def predict_rating(average, subscriber_bias, film_bias, subscriber_taste, film_taste):
    taste_match = (subscriber_taste * film_taste).sum(axis=-1)
    return average + subscriber_bias + film_bias + taste_match

In [ ]:
#@title ▶ Train on the known stars, score it, show the row { display-mode: "form" }
embeddings = train_embeddings(predict_rating, k=20, epochs=20)     # ~5 s
plot_training(embeddings, title="Learned embeddings: error on seen vs hidden ratings")
embedding_stars = embeddings.scores()
board.add("Learned embeddings + biases (k = 20)",
          rmse=rmse(test["rating"], test_predictions(embedding_stars)),
          hits=hits_at_10(embedding_stars))
show_homepage(NORA, embedding_stars, "Ranked by predicted stars · for Nora", badge_format="{:.1f}★");

**Reading the result.** ✅ RMSE ≈ **0.91**, the best star predictor on the board. The
training error keeps falling while the hidden error levels off: more passes would only
memorise. Ranked by stars, though, Nora's row is a **critics' list** (*Schindler's List*,
*Casablanca*): films she would rate highly *if* she watched them, not what she will pick.
Hits@10 ≈ **0.6**.

### "More like this", now from the learned taste scores

Part 3 called two films similar when **the same people rated them the same way**. The
embeddings give a second definition: two films are similar when their **20 learned numbers
point the same way** (cosine similarity on the columns of $Q$). For a film with thousands of
ratings the two should agree. The interesting case is the film from "Where neighbours break":
5 ratings in all, 3 of them in the 80% we learn from, so almost no shared viewers to compare,
yet 20 numbers that were nudged towards the films its few viewers also rated.

In [ ]:
#@title ▶ Compare the two definitions of "similar", for a famous film and a niche one { display-mode: "form" }
embedding_similarity = cosine_similarity(embeddings.film_taste)        # films × films, from 20 learned numbers each
compare_neighbours(["Star Wars (1977)", films.loc[niche_id, "title"]],
                   {"From who rated what (Part 3)": film_similarity,
                    "From the learned taste scores (4c)": embedding_similarity},
                   subtitle="number = similarity · left: shared viewers · right: cosine between the two films' 20 taste scores");

**Reading the result.** For *Star Wars* (583 ratings) the two lists agree at the top: the
two sequels and *Raiders* either way. Lower down the embedding list admits a little-rated
film whose 20 numbers happen to point the same way (*A Chef in Love*): cosine compares
directions and ignores how *well* a film's taste is known. For the niche film the lists have
nothing in common. The rating-based one is the mix from before (horror, Disney, slapstick);
the embedding-based one is a coherent shelf of mid-90s independent dramas, with lower
similarities (0.65 against 0.94): thin evidence stays thin, but embeddings *spread* it,
because a film's 20 numbers are shaped by every rating of every subscriber who touched it,
not only by direct overlaps. That is why factorization is the usual engine behind "More like
this" for the long tail, and why a well-known-films rule still sits in front of it.

## Part 5 — Your own homepage, both ways

Be a new Flix subscriber. The 40 most-rated films; `find_film("…")` finds any other in the
catalogue (films up to 1998).

In [ ]:
#@title ▶ The 40 most-rated films { display-mode: "form" }
film_menu()

## 🎯 Exercise 10. Rate some films

**🎯 Task.** Fill `my_ratings` with **at least 5 films** you have seen, exact titles (quotes
included), 1★–5★. Include one or two you disliked. The ratings were collected in **1997–98**,
so pick films released **up to 1998**: anything newer is not in the catalogue, and
`find_film("…")` tells you whether a title is.

### 🎯 Solution

In [ ]:
# 🎯 Exercise 10: your own ratings.
my_ratings = {
    "Star Wars (1977)": 5,
    "Raiders of the Lost Ark (1981)": 5,
    "Toy Story (1995)": 4,
    "Fargo (1996)": 4,
    "Jurassic Park (1993)": 4,
    "Titanic (1997)": 2,
    "Sense and Sensibility (1995)": 2,
    "Pulp Fiction (1994)": 1,
}

In [ ]:
#@title ▶ Your two homepages { display-mode: "form" }
my_homepages(my_ratings, film_similarity, predict_from_neighbours, film_scores)

💭 **Which row feels more like you?** Change one film to 1★ and re-run: the item-item row
moves (it reads stars), the SVD row doesn't (the watch table only knows what you *chose*).
Which behaviour do you want, for which row?

Five ratings place a new subscriber on the axes instantly, no retraining. **Zero** ratings
place nothing: you have to *try* films and learn from the clicks, which is this afternoon's
**bandits**.

## Part 6 — Ship it

In [ ]:
#@title 📊 The leaderboard { display-mode: "form" }
board.show()

**Reading the scoreboard.**

| Method | Learns from | RMSE ⭐ | Hits@10 🏠 | Good for |
|---|---|---|---|---|
| Overall average | stars | 1.13 | | the floor |
| Rank by the film's average rating | stars | 1.02 | 0.00 | nothing on its own |
| Rank by average rating, ≥ 50 ratings | stars | | 0.66 | an "acclaimed" shelf |
| Rank by most watched | choices | | 1.36 | today's row; the bar |
| Item-item neighbours | stars | 0.93 | 0.38 | **"More like this"** |
| Item-item + rule (≥ 50 ratings) | stars | | 0.84 | a safe, explainable personal row |
| SVD, watch table (k = 15) | choices | | **2.61** | **"Top picks for you"** |
| NMF, watch table (k = 15) | choices | | 2.4 | the same row, with nameable axes |
| Learned embeddings + biases | stars | **0.91** | 0.62 | **star badges** |

**Recommendation to Mira:** ship the SVD row as "Top picks for you", keep item-item for "More
like this", and take star badges from the embeddings with biases.

## Summary

1. **Score on hidden data before you ship.** Hide 20%, learn from 80%, judge on the 20%.
2. **Decide the question before the model.** Stars and homepage have different scores and
   different winners; the best star predictor made one of the worst homepages.
3. **Score a baseline, expect a rule.** "Most watched" beat the personal row; a
   well-known-films rule made it a product again, at a price in catalogue breadth.
4. **What people choose to watch is information.** A table of 1s and 0s made the best
   homepage.
5. **The table and the question matter more than the method.** SVD and NMF on choices →
   the homepage; embeddings with biases on stars → badges. The number of axes is a business choice: too few miss tastes, too many memorise.

### ⭐ Bonus A. GMF and GMF + MLP: does a neural network do better?

Everything in Part 4 ends in a dot product: products axis by axis, added **with equal
weight**. A neural network keeps the products and *learns* the rest. Both models below learn
from the known stars, on the same 80%, scored on the same 20%.

**GMF** (*Generalized Matrix Factorization*), three parts:

1. **Two embedding tables**, one row of $k$ numbers per subscriber and per film. These *are*
   $P$ and $Q$, learned as in 4c.
2. **The interaction:** the two vectors multiplied element by element, $k$ products.
3. **An output unit** that adds the products with **learned weights** $w$ and a bias, then
   squashes the sum to 0–1 with a *sigmoid*; × 5 gives stars. With all weights 1, GMF *is*
   the dot product.

**GMF + MLP** (Neural Collaborative Filtering, He et al., 2017) adds a second path: the two
vectors side by side ($2k$ numbers) through two small layers of "neurons" (a weighted sum,
negatives set to 0), which can learn rules a product cannot, such as "high on axis 1 *and*
low on axis 2". Both paths feed the same output unit. One worked example with two axes:

In [ ]:
#@title 🧩 Visualization: from two vectors to a rating, three ways (double-click to view the code) { display-mode: "form" }
gmf_explainer()

#### ⭐ Bonus Exercise. The GMF interaction

The engine builds the embedding tables, the output unit and the training loop. You write
the interaction: how the two vectors are combined.

**🎯 Task.** Return the element-wise product of the two vectors, as a Keras layer.

### 🎯 Solution

In [ ]:
# 🎯 Bonus exercise: GMF combines the two taste vectors element by element.
def gmf_interaction(subscriber_vec, film_vec):
    return layers.Multiply()([subscriber_vec, film_vec])

In [ ]:
#@title ▶ Train GMF, then GMF + MLP; score both { display-mode: "form" }
gmf = train_gmf(gmf_interaction, k=20, epochs=8)            # ~30 s on Colab
if gmf:
    gmf_stars = gmf.scores()
    board.add("GMF (k = 20)", rmse=rmse(test["rating"], test_predictions(gmf_stars)),
              hits=hits_at_10(gmf_stars))

ncf = train_ncf(k=20, mlp_dim=40, epochs=8)                 # ~40 s on Colab
if ncf:
    ncf_stars = ncf.scores()
    board.add("GMF + MLP (k = 20)", rmse=rmse(test["rating"], test_predictions(ncf_stars)),
              hits=hits_at_10(ncf_stars))
    plot_training(ncf, title="GMF + MLP: error on seen vs hidden ratings")
    show_homepage(NORA, ncf_stars, "Ranked by GMF + MLP predicted stars · for Nora", badge_format="{:.1f}★")

**Reading the result.** ✅ GMF ≈ **0.93** RMSE, GMF + MLP ≈ **0.94**: both just *behind* the
embeddings with biases (0.91), same data, same $k$. On the homepage GMF reaches about 1.0
hits, GMF + MLP 0.4; neither approaches the SVD row (2.61). The GMF + MLP hidden error
flattens after three passes while the training error keeps falling: memorising. This matches
Rendle et al. (RecSys 2020): **a well-tuned dot product with biases is hard to beat**, and
neural variants cost more to train, serve and explain.

💭 "Neural" is not a feature. Which question does it answer better, on hidden data, at what
cost?

## Appendix — How each factorization finds P and Q

Optional. Each section opens up one method from Part 4 on the toy ledger, with a slider
to step through it. Nothing below is needed to finish the exercises.

### A1. SVD: one axis at a time (Part 4a)

SVD finds the single axis that explains the most of the table, then the best second axis
for what is left, and so on. Stop after $k$ and you have the best $P$ and $Q$ with $k$ axes,
by a theorem (the Eckart–Young theorem: no other $P$ and $Q$ with $k$ axes come closer to
the table). Step through it on the toy: the rebuilt table (right) approaches the real one
(left) as axes are added.

In [ ]:
#@title 🧩 Step through SVD on the toy (use the slider) { display-mode: "form" }
show_steps(
    toy_watch, svd_steps(toy_watch, k_max=3, notes=[
        "<b>One axis.</b> Each subscriber gets one number (how much they watch), each film one number "
        "(how much it gets watched). The rebuilt table is 'popularity, for everyone': every row ranks the "
        "films in the same order. Not personal yet.",
        "<b>Two axes.</b> The second axis separates tastes: + romance, − action (User A negative, User B "
        "positive). Now the rows differ, and the blanks hold reasoned guesses: B × Movie 1 comes out low.",
        "<b>Three axes.</b> The table is rebuilt exactly, error 0. With as many axes as the table has rows, "
        "nothing is compressed and nothing is learned: this is a lookup table, not a model. The sweet spot "
        "is somewhere in between, and on the real ledger we find it by testing on the hidden ratings."]),
    title="SVD rebuilds the toy watch table axis by axis",
    subtitle="R ≈ P × Q · P: one row per subscriber · Q: one column per film · colours: darker = larger",
    row_labels=list(toy.index), col_labels=["M1 (Act)", "M2 (Act)", "M3 (Rom)", "M4 (Rom)"])

### A2. NMF: why the ratio works (Part 4b)

NMF minimises the squared error between $R$ and $PQ$ while keeping every entry ≥ 0. Take
the rule for $Q$:

$$Q_{\text{new}} = Q \;\odot\; \frac{P^{\mathsf T} R}{P^{\mathsf T} P\, Q}$$

$P^{\mathsf T} R$ is **what the data asks of $Q$**: each film's column of the table,
projected onto the current taste axes. $P^{\mathsf T} P\, Q$ is **the same quantity according
to the current model**, with $R$ replaced by its reconstruction $PQ$. Where the two agree the
ratio is 1 and the entry stays put; where the model undershoots the ratio is above 1 and the
entry grows; where it overshoots the entry shrinks. The rule is a gradient-descent step on
the squared error with a step size chosen *per entry* so that the update becomes a
multiplication, which is what keeps everything non-negative (Lee & Seung, 2001). $P$ gets
the mirror rule with the roles of rows and columns swapped. A couple of hundred rounds and
the error stops falling. Your `nmf_step` on the toy, one correction at a time:

In [ ]:
#@title 🧩 Step through YOUR NMF on the toy (use the slider) { display-mode: "form" }
_, _, nmf_toy_steps = run_nmf(nmf_step, toy_watch, k=2, iterations=100, record=(0, 1, 2, 3, 5, 10, 20, 50, 100))
show_steps(toy_watch, nmf_toy_steps, title="Your NMF corrects P and Q step by step",
           subtitle="every entry stays ≥ 0 · the error on the right falls with each update",
           row_labels=list(toy.index), col_labels=["M1 (Act)", "M2 (Act)", "M3 (Rom)", "M4 (Rom)"])

### A3. Learned embeddings: the learning loop (Part 4c)

The plainest loop there is, *stochastic gradient descent* on the known ratings:

1. start from random taste scores;
2. take one known rating $r_{u,f}$, predict it with the dot product $\hat r_{u,f} = p_u
   \cdot q_f$, and compute the error $e = r_{u,f} - \hat r_{u,f}$;
3. nudge both sides towards a smaller error: $p_u \leftarrow p_u + \eta\, e\, q_f$ and
   $q_f \leftarrow q_f + \eta\, e\, p_u$, where $\eta$ is a small *learning rate*;
4. repeat for every known rating; one full pass is an *epoch*; run many.

Step 3 moves the subscriber's taste scores towards the film's profile when the model
underestimated the rating ($e > 0$) and away from it when it overestimated. Blanks are never
looked at, so their predictions come purely from the patterns in the known cells. On the
toy: six single updates, then whole passes.

In [ ]:
#@title 🧩 Step through the learning loop on the toy stars (use the slider) { display-mode: "form" }
show_steps(toy, sgd_steps(toy, k=2, learning_rate=0.05, row_labels=list(toy.index),
                          col_labels=["Movie 1", "Movie 2", "Movie 3", "Movie 4"]),
           title="Predict one rating, compare, nudge: P and Q learn from the known stars only",
           subtitle="faded cells in P × Q = predictions for blanks (never used for learning) · error = RMSE on the 9 known cells",
           row_labels=list(toy.index), col_labels=["M1 (Act)", "M2 (Act)", "M3 (Rom)", "M4 (Rom)"],
           fmt="{:.1f}★", target_label="R · the star ledger (blank = unknown)")